In [ ]:
# ==========================================
# 1. MESCLAR OS ADAPTADORES LoRA
# ==========================================
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os

BASE_MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
LORA_PATH = "./drive/MyDrive/deepseek-agronomy-finetuned"
MERGED_PATH = "./deepseek-agronomy-merged"

print("📥 Carregando modelo base...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("📥 Carregando adaptadores LoRA...")
model = PeftModel.from_pretrained(base_model, LORA_PATH)

print("🔄 Mesclando...")
merged_model = model.merge_and_unload()

print("💾 Salvando modelo mesclado...")
merged_model.save_pretrained(MERGED_PATH)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.save_pretrained(MERGED_PATH)
print("✅ Modelo mesclado salvo!")

# ==========================================
# 2. CONVERTER PARA GGUF E QUANTIZAR
# ==========================================
# Clonar llama.cpp (se ainda não estiver)
if not os.path.exists("llama.cpp"):
    !git clone https://github.com/ggerganov/llama.cpp
    !cd llama.cpp && mkdir -p build && cd build && cmake .. && make -j4

print("🔄 Convertendo para GGUF (F16)...")
!python llama.cpp/convert_hf_to_gguf.py {MERGED_PATH} \
    --outfile ./deepseek-agronomy-f16.gguf \
    --outtype f16

print("📉 Quantizando para Q4_K_M...")
!./llama.cpp/build/bin/llama-quantize \
    ./deepseek-agronomy-f16.gguf \
    ./deepseek-agronomy-Q4_K_M.gguf \
    Q4_K_M

print("✅ Conversão concluída!")

# ==========================================
# 3. BAIXAR O MODELO
# ==========================================
#from google.colab import files
#if os.path.exists("./deepseek-agronomy-Q4_K_M.gguf"):
#    size_gb = os.path.getsize("./deepseek-agronomy-Q4_K_M.gguf") / (1024**3)
#    print(f"📊 Tamanho: {size_gb:.2f} GB")
#    files.download('deepseek-agronomy-Q4_K_M.gguf')
#else:
 #   print("❌ Arquivo não encontrado!")
  #  !find . -name "*.gguf" -type f -exec ls -lh {} \;